# Wire Rope Defect (Training)

In [ ]:
!pip install -q ultralytics roboflow python-dotenv

In [ ]:
from dotenv import load_dotenv

load_dotenv()   # reads ROBOFLOW_API_KEY from .env, see .env.example

## Detect device (CPU, GPU)

In [ ]:
import torch, os
from ultralytics import YOLO

if torch.cuda.is_available():
    DEVICE   = 0
    GPU_NAME = torch.cuda.get_device_name(0)                               # detects the GPU 
    VRAM_GB  = torch.cuda.get_device_properties(0).total_memory / 1024**3  # detects VRAM
    AMP      = True        # AMP (Automatic Mixed Precision) float16 training (faster and uses less memory)
    BATCH    = -1          # auto batch size 
    WORKERS  = 8           # number of workers for data loading
    EPOCHS   = 150         # times of epoches
    IMGSZ    = 640         # image size scale
    print(f"GPU found: {GPU_NAME} ({VRAM_GB:.1f} GB VRAM)") # print GPU name and VRAM
else:
    DEVICE   = "cpu"        
    AMP      = False       # AMP on CPU doesnt work
    BATCH    = 4           # batch size for CPU training
    WORKERS  = min(4, os.cpu_count() or 2)
    EPOCHS   = 20
    IMGSZ    = 416         # image size scale
    print(f"No GPU available -> CPU ({os.cpu_count()} cores).")
    print("Recommendation: Change runtime -> Change runtime type -> GPU (T4)")

print(f"device={DEVICE} | batch={BATCH} | epochs={EPOCHS} | imgsz={IMGSZ} | amp={AMP}")

## Dataset
Datasets: https://universe.roboflow.com/wirerope-qfgck/wire-rope-defect-f4qyg ≈  5'900 images
          https://universe.roboflow.com/yuan-9voxc/wire-rope-defect-review    ≈ 16'500 images

In [ ]:
import os
from roboflow import Roboflow

rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])   # .evn file enter key

rf.workspace("wirerope-qfgck").project("wire-rope-defect-f4qyg") \
  .version(4).download("yolov11", location="datasets/A")

The second dataset has no version and can therefore not be downloaded directly.
Use **Fork Dataset** on its Universe page, then generate a version in your own
workspace under Versions (Auto-Orient + Resize 640, augmentation off).

In [ ]:
WORKSPACE = "YOUR_WORKSPACE"   # edit

rf.workspace(WORKSPACE).project("wire-rope-defect-review") \
  .version(1).download("yolov11", location="datasets/B")

In [ ]:
# merge both exports into one dataset, --out is the DATA path below
!python prepare_wirerope.py --src datasets/A datasets/B --out datasets/wirerope_merged

In [ ]:
from pathlib import Path

DATA = Path('datasets/wirerope_merged/data.yaml')                     # edit
# DATA = Path(r'C:\Users\Name\Projects\wirerope_merged\data.yaml')    # edit
PROJECT = 'runs'                                                      # edit

In [ ]:
# Check the dataset configuration file and update the path
import yaml, pathlib

cfg = yaml.safe_load(open(DATA))
cfg['path'] = str(pathlib.Path(DATA).resolve().parent)
yaml.safe_dump(cfg, open(DATA, 'w'), sort_keys=False, allow_unicode=True)

print(cfg)
for split in ('train', 'val'):
    d = pathlib.Path(cfg['path']) / cfg[split]
    print(split, '->', d, '|', len(list(d.glob('*'))), 'Bilder')

## Training

In [ ]:
model = YOLO('yolo26s.pt') # small model
results = model.train(
    data=DATA,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    amp=AMP,
    cache='ram' if DEVICE != 'cpu' else False,
    patience=30,
    cos_lr=True,
    project=PROJECT,
    name='wirerope',
    exist_ok=True,
)

In [ ]:
# model = YOLO('/content/drive/MyDrive/runs/wirerope/weights/last.pt')
# model.train(resume=True)

## Validated and Export

In [ ]:
best = YOLO(f'{PROJECT}/wirerope/weights/best.pt')
metrics = best.val(data=DATA, split='test', device=DEVICE, imgsz=IMGSZ)

print(f"mAP50    {metrics.box.map50:.3f}")
print(f"mAP50-95 {metrics.box.map:.3f}")
for i, c in enumerate(metrics.box.ap_class_index):
    print(f"{metrics.names[c]:<20} mAP50={metrics.box.ap50[i]:.3f}")

In [ ]:
best.export(format='onnx', imgsz=IMGSZ)     # onnx export

## Plot for Documentation

In [ ]:
# Plots that Ultralytics saved automatically during training
from pathlib import Path
from IPython.display import Image, display, Markdown

RUN = Path(PROJECT) / 'wirerope'

PLOTS = {
    'results.png':                    'Loss and metric curves over all epochs',
    'confusion_matrix_normalized.png':'Which classes get confused with which (rows = true, cols = predicted)',
    'BoxPR_curve.png':                'Precision vs. recall per class - area under the curve is the mAP50',
    'BoxF1_curve.png':                'F1 score vs. confidence - the peak is the best confidence threshold',
    'labels.jpg':                     'Class balance and box size/position distribution of the training set',
    'val_batch0_pred.jpg':            'Model predictions on one validation batch',
}

for f, desc in PLOTS.items():
    p = RUN / f
    if not p.exists():                       # older versions use P_curve.png / PR_curve.png
        p = RUN / f.replace('Box', '')
    if p.exists():
        display(Markdown(f"**{p.name}** - {desc}"))
        display(Image(filename=str(p)))
    else:
        print(f"missing: {f}")

In [ ]:
# Custom curves from results.csv: losses left, metrics right
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(RUN / 'results.csv')
df.columns = df.columns.str.strip()          # older versions pad the headers with spaces

loss_cols   = [c for c in df.columns if c.endswith('_loss')]
metric_cols = [c for c in df.columns if c.startswith('metrics/')]

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

for c in loss_cols:                          # train = solid, val = dashed
    ax[0].plot(df['epoch'], df[c], ls='--' if c.startswith('val/') else '-', label=c)
ax[0].set(xlabel='epoch', ylabel='loss', title='Training vs. validation loss')
ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)

for c in metric_cols:
    ax[1].plot(df['epoch'], df[c], label=c.replace('metrics/', ''))
ax[1].set(xlabel='epoch', ylabel='score', ylim=(0, 1), title='Precision / recall / mAP')
ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)

plt.tight_layout(); plt.show()

best_ep = df['metrics/mAP50-95(B)'].idxmax()
print(f"best epoch: {int(df.loc[best_ep, 'epoch'])}  "
      f"mAP50={df.loc[best_ep, 'metrics/mAP50(B)']:.3f}  "
      f"mAP50-95={df.loc[best_ep, 'metrics/mAP50-95(B)']:.3f}")

In [ ]:
# Custom curves from results.csv: losses left, metrics right
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(RUN / 'results.csv')
df.columns = df.columns.str.strip()          # older versions pad the headers with spaces

loss_cols   = [c for c in df.columns if c.endswith('_loss')]
metric_cols = [c for c in df.columns if c.startswith('metrics/')]

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

for c in loss_cols:                          # train = solid, val = dashed
    ax[0].plot(df['epoch'], df[c], ls='--' if c.startswith('val/') else '-', label=c)
ax[0].set(xlabel='epoch', ylabel='loss', title='Training vs. validation loss')
ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)

for c in metric_cols:
    ax[1].plot(df['epoch'], df[c], label=c.replace('metrics/', ''))
ax[1].set(xlabel='epoch', ylabel='score', ylim=(0, 1), title='Precision / recall / mAP')
ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)

plt.tight_layout(); plt.show()

best_ep = df['metrics/mAP50-95(B)'].idxmax()
print(f"best epoch: {int(df.loc[best_ep, 'epoch'])}  "
      f"mAP50={df.loc[best_ep, 'metrics/mAP50(B)']:.3f}  "
      f"mAP50-95={df.loc[best_ep, 'metrics/mAP50-95(B)']:.3f}")

In [ ]:
# Predictions on a few test images
import random
from pathlib import Path
import matplotlib.pyplot as plt

test_dir = Path(cfg['path']) / cfg.get('test', 'images/test')
imgs = sorted(p for p in test_dir.glob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'})
sample = random.Random(0).sample(imgs, min(6, len(imgs)))

preds = best.predict(sample, imgsz=IMGSZ, conf=0.25, device=DEVICE, verbose=False)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, r in zip(axes.ravel(), preds):
    ax.imshow(r.plot()[:, :, ::-1])           # plot() returns BGR
    ax.set_title(f"{Path(r.path).name[:28]}  ({len(r.boxes)} det)", fontsize=8)
    ax.axis('off')
for ax in axes.ravel()[len(preds):]:
    ax.axis('off')
plt.tight_layout(); plt.show()